# B2-019 — Practice p12

**Set:** B · **Type:** constrained-coding · **Difficulty:** core · **Minutes:** 50

**Concepts:** transformer-residual-layernorm, position-wise-feed-forward, transformer-block

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260808`  
**Qualified Book 1 prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C11-neural-training`  
**Remediation:** review the linked Book 1 units before continuing: [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb).

## Task

Implement `PreNormBlock(torch.nn.Module)`. Its constructor is exactly `__init__(self, attention, width=4)`, stores the supplied module as `self.attention`, creates affine `nn.LayerNorm(width, dtype=torch.float64)` modules named `norm1` and `norm2`, and creates `self.ffn = nn.Sequential(nn.Linear(4,8), nn.ReLU(), nn.Linear(8,4))` in `float64`. Its `forward(x, allowed)` uses exactly `y=x+self.attention(self.norm1(x),allowed)` and `z=y+self.ffn(self.norm2(y))`, then returns `z`.

Use the supplied `x`, Boolean `allowed`, and `SuppliedAttention`. For the fixed probe's zero-sublayer case, zero the attention projection and both FFN linear layers and prove `z == x`. For the nonzero probe, call `initialize_nonzero(block)` and return `nonzero_z`; compare against a direct evaluation of the two stated recurrences. The LayerNorm affine parameters remain their default ones/zeros in both probes.

Pinned contract: seed `20260808`; CPU dtype `float64`; `B=2,N=3,D=4`; exact input and all projection/FFN weights and biases are supplied below; output shape `(2,3,4)`; `atol=1e-10`, `rtol=1e-10`. Allowed APIs: `nn.Module`, `nn.LayerNorm`, `nn.Linear`, `nn.ReLU`, `nn.Sequential`, and supplied tensor setup. Banned APIs: `nn.TransformerEncoderLayer`, post-norm recurrence, CUDA/MPS, file/network access.

In [ ]:
import torch
from torch import nn

torch.manual_seed(20260808)
x = torch.arange(24, dtype=torch.float64).reshape(2, 3, 4) / 10.0
allowed = torch.tril(torch.ones(2, 3, 3, dtype=torch.bool))
ATTENTION_WEIGHT = torch.tensor(
    [[1.0, 0.0, 0.0, 0.0],
     [0.0, 0.5, 0.0, 0.0],
     [0.0, 0.0, -0.5, 0.0],
     [0.25, 0.0, 0.0, 0.75]],
    dtype=torch.float64,
)
FFN_WEIGHT_1 = torch.arange(32, dtype=torch.float64).reshape(8, 4) / 50.0 - 0.3
FFN_BIAS_1 = torch.linspace(-0.2, 0.2, 8, dtype=torch.float64)
FFN_WEIGHT_2 = torch.arange(32, dtype=torch.float64).reshape(4, 8) / 80.0 - 0.15
FFN_BIAS_2 = torch.tensor([0.05, -0.05, 0.1, -0.1], dtype=torch.float64)


class SuppliedAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.projection = nn.Linear(4, 4, bias=False, dtype=torch.float64)

    def forward(self, normalized_x, allowed):
        if allowed.dtype != torch.bool or allowed.shape != (2, 3, 3):
            raise ValueError("allowed must be Boolean with shape (2,3,3)")
        return self.projection(normalized_x)


def initialize_nonzero(block):
    with torch.no_grad():
        block.attention.projection.weight.copy_(ATTENTION_WEIGHT)
        block.ffn[0].weight.copy_(FFN_WEIGHT_1)
        block.ffn[0].bias.copy_(FFN_BIAS_1)
        block.ffn[2].weight.copy_(FFN_WEIGHT_2)
        block.ffn[2].bias.copy_(FFN_BIAS_2)

In [ ]:
# Implement PreNormBlock and run the zero and nonzero probes here.


## Your response

Show the requested derivation, implementation, audit, or justification here.